# 02 — Détection d'anomalies

Ce notebook explore la détection d'anomalies par Isolation Forest + LOF et évalue les résultats vs labels fraude réels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
sys.path.insert(0, '../src')

from load_data import load_features
from preprocess import preprocess
from reduce_dimension import run_reduction
from anomaly_detection import run_anomaly_detection
from merge_results import run_merge

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
df_features = load_features()
X_scaled, cols, _ = preprocess(df_features)
df_3d = run_reduction(X_scaled, df_features['node_id'])

In [ ]:
df_anomalies = run_anomaly_detection(X_scaled, df_features['node_id'], df_features)
print(df_anomalies['risk_level'].value_counts())
df_anomalies[['node_id', 'anomaly_score', 'risk_level', 'is_fraud_node']].head(20)

In [ ]:
# Distribution des scores d'anomalie
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_anomalies['anomaly_score'], bins=50, color='coral', edgecolor='black')
axes[0].set_title('Distribution anomaly_score')
axes[0].axvline(0.4, color='orange', linestyle='--', label='seuil suspect')
axes[0].axvline(0.7, color='red', linestyle='--', label='seuil critique')
axes[0].legend()

axes[1].hist(df_anomalies['if_raw_score'], bins=50, color='steelblue', edgecolor='black')
axes[1].set_title('Isolation Forest raw score')

axes[2].hist(df_anomalies['lof_score'], bins=50, color='seagreen', edgecolor='black')
axes[2].set_title('LOF score')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation spatiale : anomalies dans l'espace 3D (projection x,y)
merged = df_3d.merge(df_anomalies[['node_id', 'anomaly_score', 'risk_level', 'is_fraud_node']], on='node_id')

plt.figure(figsize=(12, 8))
colors = {'normal': 'steelblue', 'suspect': 'orange', 'critique': 'red'}
for level, color in colors.items():
    sub = merged[merged['risk_level'].astype(str) == level]
    plt.scatter(sub['x'], sub['y'], c=color, s=5, alpha=0.6, label=level)
plt.title('Nœuds par niveau de risque (projection UMAP x,y)')
plt.legend(markerscale=4)
plt.show()

In [ ]:
# Matrice de confusion anomalie vs fraude réelle
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

if 'is_fraud_node' in df_anomalies.columns:
    cm = confusion_matrix(df_anomalies['is_fraud_node'], df_anomalies['anomaly_label'])
    print(classification_report(df_anomalies['is_fraud_node'], df_anomalies['anomaly_label'],
                                 target_names=['Normal', 'Anomalie']))
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Prédit Normal', 'Prédit Anomalie'],
                yticklabels=['Réel Normal', 'Réel Fraude'])
    plt.title('Matrice de confusion : Anomalie vs Fraude réelle')
    plt.tight_layout()
    plt.show()

In [ ]:
# Top 20 nœuds les plus anormaux
top_anomalies = df_anomalies.nlargest(20, 'anomaly_score')[
    ['node_id', 'anomaly_score', 'risk_level', 'is_fraud_node']
]
print('Top 20 nœuds anormaux :')
print(top_anomalies.to_string())